# 07 – Phân cụm: K-Means, Agglomerative, DBSCAN/HDBSCAN, t-SNE

**Người thực hiện:** thinh (Nguyễn Vĩnh Thịnh)  
**Mục tiêu:** Trả lời RQ3: phân cụm không giám sát tái tạo cấu trúc ngành hàng tới đâu?

**Quy tắc:** chạy TỪNG ô một, đổi tham số rồi chạy lại; mỗi ngày làm xong thì `git add` + `git commit -m "<mô tả đúng việc đã làm>"` + `git push`. Cuối buổi xuất HTML: `python ../scripts/export_notebook_html.py <notebook> thinh`.

**Các bước:**
1. K-Means: quét K (6–20) và số chiều SVD.
2. Agglomerative (mẫu 5.000–8.000 điểm): linkage ward/average/complete.
3. DBSCAN: quét eps, min_samples; HDBSCAN: min_cluster_size.
4. t-SNE trên mẫu 5.000 điểm, tô màu theo ngành và theo cụm.
5. Chọn thuật toán tốt nhất, giải thích Silhouette/ARI/NMI bằng lời.

In [1]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.append('../src')
from run_experiment import run
from figlog import save_fig
from exp_logger import log_generic
from common import load_data, clean_title


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

df = load_data(); df['clean'] = df.title.map(clean_title); y = df.category_label.values
Xt = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b[\w\-\.\+]+\b").fit_transform(df.clean)
def embed(m):
    return Normalizer().fit_transform(TruncatedSVD(m, random_state=42).fit_transform(Xt))

def score(Z, lab, algo, **cfg):
    rng = np.random.RandomState(42); s = rng.choice(len(Z), min(10000, len(Z)), replace=False)
    ok = lab[s] != -1  # bỏ điểm nhiễu của DBSCAN khi tính silhouette
    r = dict(algo=algo, **cfg, n_clusters=len(set(lab)) - (1 if -1 in lab else 0), noise_pct=float((lab == -1).mean()),
             silhouette=silhouette_score(Z[s][ok], lab[s][ok], metric='cosine') if len(set(lab[s][ok])) > 1 else float('nan'),
             ari=adjusted_rand_score(yy, lab), nmi=normalized_mutual_info_score(yy, lab))
    log_generic('thinh', 'cluster_log', r); print(r); return r

## 1. K-Means

In [5]:
res = []
for k in range(6, 21, 2):
    lab = KMeans(k, n_init=10, random_state=42).fit_predict(Z)
    res.append(score(Z, lab, 'kmeans', K=k, svd=50))
ket_qua = pd.DataFrame(res)[['K', 'silhouette', 'ari', 'nmi']]
ket_qua

{'algo': 'kmeans', 'K': 6, 'svd': 50, 'n_clusters': 6, 'noise_pct': 0.0, 'silhouette': 0.21560822546957853, 'ari': 0.38795468553117646, 'nmi': 0.6171020800332649}
{'algo': 'kmeans', 'K': 8, 'svd': 50, 'n_clusters': 8, 'noise_pct': 0.0, 'silhouette': 0.26888575323135444, 'ari': 0.4831093153719097, 'nmi': 0.7167283195830451}
{'algo': 'kmeans', 'K': 10, 'svd': 50, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.3016820669038938, 'ari': 0.6229551874853975, 'nmi': 0.7318251931951658}
{'algo': 'kmeans', 'K': 12, 'svd': 50, 'n_clusters': 12, 'noise_pct': 0.0, 'silhouette': 0.30791268597888705, 'ari': 0.567926709838571, 'nmi': 0.7111196133879203}
{'algo': 'kmeans', 'K': 14, 'svd': 50, 'n_clusters': 14, 'noise_pct': 0.0, 'silhouette': 0.31343436497090404, 'ari': 0.5444062090302868, 'nmi': 0.7011448477035328}
{'algo': 'kmeans', 'K': 16, 'svd': 50, 'n_clusters': 16, 'noise_pct': 0.0, 'silhouette': 0.3377260258364998, 'ari': 0.5459624658142548, 'nmi': 0.7040620681187659}
{'algo': 'kmeans', 'K'

,K,silhouette,ari,nmi
0,6,0.215608,0.387955,0.617102
1,8,0.268886,0.483109,0.716728
2,10,0.301682,0.622955,0.731825
3,12,0.307913,0.567927,0.711120
4,14,0.313434,0.544406,0.701145
5,16,0.337726,0.545962,0.704062
6,18,0.344808,0.551461,0.695845
7,20,0.347170,0.463338,0.660763


In [3]:
Z = embed(50); yy = y
lab = KMeans(10, n_init=10, random_state=42).fit_predict(Z)
score(Z, lab, 'kmeans', K=10, svd=50)

{'algo': 'kmeans', 'K': 10, 'svd': 50, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.3016820669038938, 'ari': 0.6229551874853975, 'nmi': 0.7318251931951658}


{'algo': 'kmeans',
 'K': 10,
 'svd': 50,
 'n_clusters': 10,
 'noise_pct': 0.0,
 'silhouette': 0.3016820669038938,
 'ari': 0.6229551874853975,
 'nmi': 0.7318251931951658}

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/thinh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính:0.2936
> - Nhận xét:0.5636
> - Lần sau thử:k=8,K=12

## 2. Agglomerative (trên mẫu)

In [ ]:
idx = np.random.RandomState(0).choice(len(df), 6000, replace=False)
Zs, yy = embed(50)[idx], y[idx]
lab = AgglomerativeClustering(10, linkage='ward').fit_predict(Zs)
score(Zs, lab, 'agglomerative', K=10, linkage='ward', n_sample=6000)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/thinh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 3. DBSCAN / HDBSCAN

In [ ]:
lab = DBSCAN(eps=0.3, min_samples=10, metric='euclidean').fit_predict(Zs)
score(Zs, lab, 'dbscan', eps=0.3, min_samples=10, n_sample=6000)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/thinh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

In [ ]:
lab = HDBSCAN(min_cluster_size=30).fit_predict(Zs)
score(Zs, lab, 'hdbscan', min_cluster_size=30, n_sample=6000)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/thinh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 4. t-SNE

In [ ]:
from sklearn.manifold import TSNE
P = TSNE(2, perplexity=30, random_state=42, init='pca').fit_transform(Zs)
fig, ax = plt.subplots(figsize=(7, 6))
for c in np.unique(yy):
    ax.scatter(*P[yy == c].T, s=3, label=c)
ax.legend(markerscale=4, fontsize=7); ax.set_title('t-SNE (mẫu 6.000) tô màu theo ngành')
save_fig(fig, 'thinh_tsne_category', notebook='07_Thinh_PhanCum.ipynb', member='thinh', cell='Mục 4', y_nghia='(tự viết)')

5 Kết luận


### 5. Kết luận
- **Số cụm K tốt nhất:** Dựa trên quá trình quét K từ 6 đến 20 với SVD-50, **K = 10** là giá trị tối ưu nhất vì đạt điểm NMI cao nhất (`0.7318`), đồng thời chỉ số ARI cũng đạt mức cao nhất (`0.623`).
- **Đánh giá NMI sau khi sửa TF-IDF:** Sau khi bổ sung đúng `token_pattern` cho `TfidfVectorizer`, chỉ số NMI tại K=10 đã tăng lên và phục hồi chính xác bằng mức `0.7318` của bản giữa kỳ (đúng như kỳ vọng đặt ra).
- **So sánh giữa Silhouette và NMI/ARI:** 
  - Chỉ số **Silhouette** tăng dần theo số cụm lớn (đạt cao nhất ở $K=20$ với giá trị `0.347`), do đặc thù không gian vector văn bản thưa thớt khiến Silhouette có xu hướng ưa chuộng việc phân chia thành nhiều cụm nhỏ hơn.
  - Ngược lại, **NMI** và **ARI** (các chỉ số đánh giá có nhãn ground-truth) đạt đỉnh rõ rệt tại $K=10$ rồi giảm dần. Do đó, $K=10$ là lựa chọn chính xác nhất về mặt ngữ nghĩa phân cụm thực tế so với nhãn gốc của tập dữ liệu.